# ML-KEM SimpleSerial 테스트 (CW308T-STM32F4HWC)

`firmware/mcu/simpleserial-mlkem` 펌웨어(pqm4 ML-KEM 구현)를 빌드·플래시하고,
호스트의 순수 Python ML-KEM(`kyber-py`)과 복호화 결과를 비교한 뒤 파형을 수집합니다.

순서: **1. 설정 → 2. 연결 → 3. 빌드·플래시 → 4. 기능 검증 → 5. 파형 1개 → 6. 파형 대량 수집**

필요 패키지: `pip install chipwhisperer kyber-py numpy matplotlib`

## 1. 설정

In [ ]:
SCOPETYPE = "OPENADC"          # CW-Lite / CW-Pro / Husky
PLATFORM  = "CW308_STM32F4"    # CW308T-STM32F4HWC (STM32F415)도 동일
SS_VER    = "SS_VER_1_1"

PARAMS    = 768                # 512 | 768 | 1024
IMPL      = "m4fstack"         # "m4fspeed" | "m4fstack" | "clean"
TRIGGER   = 1                  # 1: 소스의 #ifdef MLKEM_TRIGGER 위치에서 트리거 (현재 768 m4fstack indcpa_dec 첫 basemul) / 0: 트리거 없음
OPT       = "3"                # 컴파일 최적화: "3" = -O3 (pqm4 기본, 논문 비교용) / "s" = -Os (CW 기본)
N_TRACES  = 1000               # 6단계에서 수집할 파형 수

## 2. ChipWhisperer 연결

ChipWhisperer 강의 노트북과 같은 방식으로 `Setup_Scripts/Setup_Generic.ipynb`를 실행해 `scope`, `target`, `prog`, `reset_target(scope)`를 만듭니다. `default_setup()`이 7.37MHz 타깃 클럭, ADC = 4배 클럭, UART/nRST 핀을 설정합니다.

In [ ]:
%run "Setup_Scripts/Setup_Generic.ipynb"

## 3. 빌드와 플래시

CW 강의 노트북처럼 `%%bash -s`로 설정값을 넘겨 `make`를 실행하고, 만들어진 hex를 플래시합니다.

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER" "$PARAMS" "$IMPL" "$TRIGGER" "$OPT"
cd ../firmware/mcu/simpleserial-mlkem
make PLATFORM=$1 SS_VER=$2 PARAMS=$3 IMPL=$4 TRIGGER=$5 OPT=$6 -j

In [ ]:
fw_path = "../firmware/mcu/simpleserial-mlkem/simpleserial-mlkem{}-{}-{}.hex".format(
    PARAMS, IMPL, PLATFORM)
cw.program_target(scope, prog, fw_path)
reset_target(scope)

## 4. 기능 검증

펌웨어 명령 (SimpleSerial 1.1, 고정 길이)

| 명령 | 페이로드 | 동작 |
|---|---|---|
| `s` | 129바이트 = 조각 번호 1 + 데이터 128 | 비밀키(2400바이트) 조각 적재 |
| `c` | 129바이트 = 조각 번호 1 + 데이터 128 | 암호문(1088바이트) 조각 적재 |
| `d` | 0바이트 | 트리거 ↔ `crypto_kem_dec`, 결과를 `r`로 32바이트 반환 |
| `x` | 0바이트 | 버퍼 초기화 |
| `i` | 0바이트 | `r` 4바이트: 파라미터/256, 구현(1=m4fstack, 2=m4fspeed, 3=clean), 트리거(0=없음, 1=있음), 조각 크기 |

In [ ]:
from kyber_py.ml_kem import ML_KEM_512, ML_KEM_768, ML_KEM_1024
KEM = {512: ML_KEM_512, 768: ML_KEM_768, 1024: ML_KEM_1024}[PARAMS]

CHUNK = 128

def send_chunks(cmd, data):
    for i in range(0, len(data), CHUNK):
        part = data[i:i + CHUNK].ljust(CHUNK, b"\x00")      # SS 1.1은 고정 길이
        target.simpleserial_write(cmd, bytes([i // CHUNK]) + part)
        if target.simpleserial_wait_ack(timeout=500) is None:
            raise RuntimeError(f"'{cmd}' 조각 {i // CHUNK}: ack 없음")

def load_key(dk):
    send_chunks("s", dk)

def decaps(ct, capture=False):
    send_chunks("c", ct)
    if capture:
        scope.arm()
    target.simpleserial_write("d", bytearray())
    if capture and scope.capture():
        print("경고: 캡처 타임아웃")
    ss = target.simpleserial_read("r", 32, timeout=3000)
    return None if ss is None else bytes(ss)

# 펌웨어가 설정과 같은지 확인
target.simpleserial_write("i", bytearray())
fw = target.simpleserial_read("r", 4, timeout=1000)
impl_name = {1: "m4fstack", 2: "m4fspeed", 3: "clean"}
print("firmware:", fw[0] * 256, impl_name.get(fw[1]), "trigger", fw[2])
assert (fw[0] * 256, impl_name.get(fw[1]), fw[2]) == (PARAMS, IMPL, TRIGGER), "플래시된 펌웨어가 설정과 다릅니다"

ek, dk = KEM.keygen()
load_key(dk)
for i in range(5):
    key, ct = KEM.encaps(ek)
    if i == 4:                                            # 변조된 암호문 -> implicit rejection
        ct = bytes([ct[0] ^ 1]) + ct[1:]
        key = KEM.decaps(dk, ct)
    ss = decaps(ct)
    print(i, "OK" if ss == key else f"MISMATCH {ss}")

## 5. 파형 1개 수집

`trig_count`(트리거가 high였던 ADC 샘플 수)를 보고 `scope.adc.samples`를 정합니다. ADC가 타깃 클럭의 4배라 구간 샘플 수는 대략 사이클 × 4입니다. CW-Lite는 최대 약 24k 샘플입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

scope.adc.samples = 24400 if scope._is_husky is False else 100000
key, ct = KEM.encaps(ek)
ss = decaps(ct, capture=True)
trace = scope.get_last_trace()
print("shared secret match:", ss == key)
print("trigger length (samples):", scope.adc.trig_count)

plt.figure(figsize=(14, 3)); plt.plot(trace, lw=0.5); plt.title(f"ML-KEM-{PARAMS} {IMPL} decaps"); plt.show()

## 6. 파형 대량 수집

고정 비밀키, 무작위 암호문으로 수집해 `traces/`에 저장합니다. CPA 대상이 basemul이면 `scope.adc.samples`를 위 `trig_count` 근처로 줄이세요.

In [ ]:
from tqdm.auto import tqdm

import pathlib
out = pathlib.Path("../traces"); out.mkdir(exist_ok=True)
traces, cts, keys = [], [], []
for _ in tqdm(range(N_TRACES)):
    key, ct = KEM.encaps(ek)
    ss = decaps(ct, capture=True)
    if ss != key:
        continue
    traces.append(scope.get_last_trace())
    cts.append(np.frombuffer(ct, dtype=np.uint8))
    keys.append(np.frombuffer(key, dtype=np.uint8))

stamp = time.strftime("%Y%m%d-%H%M%S")
np.savez_compressed(out / f"mlkem{PARAMS}-{IMPL}-{stamp}.npz",
                    traces=np.array(traces, dtype=np.float32), ct=np.array(cts), ss=np.array(keys),
                    dk=np.frombuffer(dk, dtype=np.uint8), ek=np.frombuffer(ek, dtype=np.uint8))
print(len(traces), "traces saved")

In [ ]:
scope.dis()
target.dis()